# step 0: import libraries

In [1]:
import pandas as pd
from pathlib import Path
import os
import ast
import math
import numpy as np

# set display options
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

# Step 1: load data

In [2]:
dir = os.getcwd()
data_path = Path(dir).parent.parent / 'data' / 'scotus_raw.csv'
print(data_path)

df = pd.read_csv(data_path)

/Users/ellahoang-simon/Documents/GitHub/thesis-continual-learning/data/scotus_raw.csv


/var/folders/vv/65ktdvqj4w1126gwtq_ft3f00000gn/T/ipykernel_99078/120808531.py:5: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


# step 2; normalize

In [3]:
# normalize case_name and correct date datatype
df["case_name"] = (
    df["case_name"]
      .astype(str)
      .str.strip()
      .str.replace(r"\s+", " ", regex=True)  # collapse weird multiple spaces
      .str.lower()
)
df["date_filed"] = pd.to_datetime(df["date_filed"], errors="coerce")

# step 3. Parse opinions into real Python objects

In [4]:
# parse opinions column
def to_opinion_list(cell):
    # already a list?
    if isinstance(cell, list):
        return cell

    # missing?
    if cell is None or (isinstance(cell, float) and math.isnan(cell)):
        return []

    # string like "[{'author_str': ..., 'opinion_text': '...'}]"
    if isinstance(cell, str):
        s = cell.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return []

    # fallback
    return []

df["opinions_parsed"] = df["opinions"].apply(to_opinion_list)


# 4. Build a canonical opinion_text column + its length

In [5]:
def merge_full_opinion_text(op_list):
    parts = []
    for op in op_list:
        if not isinstance(op, dict):
            continue
        body = op.get("opinion_text")
        if not body or not isinstance(body, str):
            body = op.get("text")
        if body and isinstance(body, str):
            parts.append(body.strip())
    return "\n\n".join(parts).strip()

df["opinion_text"] = df["opinions_parsed"].apply(merge_full_opinion_text)
df["opinion_len"] = df["opinion_text"].str.len().fillna(0).astype(int)

df[["case_name", "date_filed", "opinion_len"]].head()


,case_name,date_filed,opinion_len
0,"tolbert v. moore, secretary, florida department of corrections",2002-04-15,210
1,united states v. rauscher,1886-12-06,124740
2,iheke v. united states,2002-06-24,201
3,mcgrath v. manufacturers trust co.,1949-11-07,18417
4,in re beaumont,2002-04-22,174


In [6]:
# double check
df[["case_name", "date_filed", "opinion_len"]].head()
print(df["opinion_text"].iloc[0][:500])

535 U.S. 1000
    TOLBERTv.MOORE, SECRETARY, FLORIDA DEPARTMENT OF CORRECTIONS.
    No. 01-8382.
    Supreme Court of the United States.
    April 15, 2002.
    
      1
      C. A. 11th Cir. Certiorari denied.


# 5. Drop exact duplicate rows

In [7]:
# drop exact duplicates = same case_name and date_filed with exact same text 
dedup_cols = ["case_name", "date_filed", "opinion_text"]

before = len(df)
df_dedup = df.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)
after = len(df_dedup)

print("Rows before removing exact clones:", before)
print("Rows after removing exact clones:", after)
print("Removed:", before - after)

Rows before removing exact clones: 498975
Rows after removing exact clones: 422351
Removed: 76624


# step 6. define metadata fields to keep

In [8]:
# # metadata columns to keep
# meta_cols = ['judges',
#             'date_filed',
#             'date_filed_is_approximate',
#             'case_name',
#             'attorneys',
#             'syllabus',
#             'summary',
#             'history',
#             'citation_count',
#             'arguments',
#             'headmatter',
#             'opinions_parsed',
#             'opinion_text',
#             'opinion_len']

meta_cols = [
    "judges",
    "date_filed_is_approximate",
    "attorneys",
    "syllabus",
    "summary",
    "history",
    "citation_count",
    "arguments",
    "headmatter",
    "opinions_parsed",     
]

# step 7. Merge rows that share the same (case_name, date_filed)

In [9]:
def combine_group(grp: pd.DataFrame) -> pd.Series:
    # 1. merge opinion_text across all rows in this (case_name, date_filed)
    merged_texts = []
    seen_texts = set()

    for txt in grp["opinion_text"]:
        if isinstance(txt, str):
            s = txt.strip()
            if s and s not in seen_texts:
                seen_texts.add(s)
                merged_texts.append(s)

    combined_text = "\n\n".join(merged_texts).strip()

    out = {
        "case_name": grp.iloc[0]["case_name"],
        "date_filed": grp.iloc[0]["date_filed"],
        "opinion_text": combined_text,
        "opinion_len": len(combined_text),
        "rows_in_group": len(grp),   # how many original rows we merged
    }

    # 2. carry forward metadata columns you want to keep
    for col in meta_cols:
        if col not in grp.columns:
            continue

        if col == "opinions_parsed":
            # Merge lists of dicts across the group
            merged_list = []
            seen_opinions = set()

            for cell in grp[col]:
                # cell should be a list of dicts, or maybe NaN/None
                if isinstance(cell, list):
                    for item in cell:
                        if isinstance(item, dict):
                            # create a hashable signature for dedupe
                            sig = tuple(sorted(item.items()))
                        else:
                            sig = repr(item)

                        if sig not in seen_opinions:
                            seen_opinions.add(sig)
                            merged_list.append(item)

            out[col] = merged_list

        else:
            # for normal metadata columns: take first non-null value
            non_null = grp[col].dropna()
            out[col] = non_null.iloc[0] if len(non_null) else None

    return pd.Series(out)


In [10]:
# apply combine_group function to each group
final_df = (
    df_dedup
      .groupby(["case_name", "date_filed"], dropna=False)
      .apply(combine_group)
      .reset_index(drop=True)
)

print(final_df.info())
print(final_df.head(3))


/var/folders/vv/65ktdvqj4w1126gwtq_ft3f00000gn/T/ipykernel_99078/2692675558.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_group)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 387078 entries, 0 to 387077
Data columns (total 15 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   case_name                  387078 non-null  object        
 1   date_filed                 387078 non-null  datetime64[ns]
 2   opinion_text               387078 non-null  object        
 3   opinion_len                387078 non-null  int64         
 4   rows_in_group              387078 non-null  int64         
 5   judges                     47609 non-null   object        
 6   date_filed_is_approximate  387078 non-null  bool          
 7   attorneys                  77570 non-null   object        
 8   syllabus                   10852 non-null   object        
 9   summary                    17278 non-null   object        
 10  history                    2290 non-null    object        
 11  citation_count             387078 non-null  int64   

# 8. Sanity checks

In [11]:
print("Rows after merge:", len(final_df))

print("Merged groups (>1 row):",
      (final_df["rows_in_group"] > 1).sum(),
      "out of",
      len(final_df))

# after merge, no (case_name, date_filed) pair should repeat:
dupe_check = (
    final_df.groupby(["case_name", "date_filed"])
            .size()
            .reset_index(name="rows_in_group")
            .query("rows_in_group > 1")
)
print("Still duplicated after merge:", len(dupe_check))

# verify we kept important columns:
print("Columns in final_df:")
print(final_df.columns.tolist())


Rows after merge: 387078
Merged groups (>1 row): 33240 out of 387078


Still duplicated after merge: 0
Columns in final_df:
['case_name', 'date_filed', 'opinion_text', 'opinion_len', 'rows_in_group', 'judges', 'date_filed_is_approximate', 'attorneys', 'syllabus', 'summary', 'history', 'citation_count', 'arguments', 'headmatter', 'opinions_parsed']


In [24]:
# taking the example of 'williams v. united states' which had duplicates we can see that they were correctly removed and these are all unique based off of their dates text, and other metadata columns
# final_df[final_df['case_name'] == 'williams v. united states']

Here we can see the cases with the same name hat occured on the same date were correctly mereged into one row. 

In [ ]:
final_df[(final_df['case_name'] == 'brown v. united states') & (final_df['date_filed'] == '2010-10-04')]

In [18]:
# final_df[(final_df['case_name'] == 'williams v. united states') & (final_df['date_filed'] == '2011-10-03')]

# 9. save final df

In [22]:
save_path = Path(dir).parent.parent / 'data' / 'scotus_final_clean.csv'
final_df.to_csv(save_path, index=False)